# Day 14: System Prompt Versioning System

## Core Theory (Just-in-Time)
As LLM applications move to production, prompts become critical infrastructure—similar to application code or database schemas. A slight modification to a system prompt can drastically alter model behavior, introduce regressions, or break downstream parsers.

### The "Why"
- **Reproducibility:** If an application starts failing, you need to know exactly which prompt was active at the time.
- **A/B Testing & Evaluation:** To compare prompt effectiveness, you must systematically track inputs, prompts, and corresponding outputs.
- **Rollbacks:** When a new prompt underperforms, you must be able to revert to a stable, known version instantly.

### The "How"
We implement prompt versioning by abstracting the prompt out of application code and into a managed registry. This can be achieved using a structured `PromptRegistry` class that relies on modern tooling like Pydantic for validation. Each prompt is treated as an immutable artifact with an explicit version.


## Code Implementation

We will create a `PromptRegistry` using Python, `pydantic` for data validation, and `langchain_core` for the prompt templates.

> Note: We use strict type hinting and docstrings as per our production-first requirements.


### Basic Implementation
Isolates the core concept with minimal boilerplate.

In [1]:
import json
from typing import Dict, List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class PromptVersion(BaseModel):
    """Represents a single, immutable version of a system prompt."""
    version: str = Field(..., description="Semantic version string (e.g., '1.0.0')")
    template_str: str = Field(..., description="The raw prompt template string")
    description: str = Field(..., description="Reason for this version or changes made")

class BasicPromptRegistry:
    """Basic Implementation: A simple registry to manage system prompts."""
    def __init__(self) -> None:
        self._registry: Dict[str, List[PromptVersion]] = {}

    def register(self, name: str, version: str, template_str: str, description: str) -> None:
        if name not in self._registry:
            self._registry[name] = []
        
        self._registry[name].append(PromptVersion(version=version, template_str=template_str, description=description))
        print(f"Registered {name} v{version}")

    def get_prompt(self, name: str, version: str) -> PromptTemplate:
        versions = self._registry.get(name, [])
        target_version = next((pv for pv in versions if pv.version == version), None)
        if not target_version:
            raise ValueError(f"Prompt {name} v{version} not found.")
        return PromptTemplate.from_template(target_version.template_str)

# Basic Example Usage
if __name__ == "__main__":
    basic_registry = BasicPromptRegistry()
    basic_registry.register("basic_agent", "1.0.0", "Answer: {question}", "Initial")
    print(basic_registry.get_prompt("basic_agent", "1.0.0").format(question="Hi?"))



Registered basic_agent v1.0.0
Answer: Hi?


### Medium Implementation
Emphasizes clean OOP, state management, and semantic versioning validation.

In [2]:
from typing import Dict, List, Optional
from pydantic import BaseModel, Field, field_validator
from langchain_core.prompts import PromptTemplate
import re

class MediumPromptVersion(BaseModel):
    """Medium Implementation: Adds semantic version validation."""
    version: str = Field(..., description="Semantic version string (e.g., '1.0.0')")
    template_str: str = Field(..., description="The raw prompt template string")
    description: str = Field(..., description="Reason for this version or changes made")

    @field_validator('version')
    @classmethod
    def check_semver(cls, v: str) -> str:
        if not re.match(r"^\d+\.\d+\.\d+$", v):
            raise ValueError("Version must be in semver format (X.Y.Z)")
        return v

class MediumPromptRegistry:
    """Medium Implementation: Emphasizes OOP state management and interactions."""
    def __init__(self) -> None:
        self._registry: Dict[str, List[MediumPromptVersion]] = {}

    def register(self, name: str, version: str, template_str: str, description: str) -> None:
        if name not in self._registry:
            self._registry[name] = []
            
        for pv in self._registry[name]:
            if pv.version == version:
                raise ValueError(f"Prompt '{name}' version '{version}' already exists.")
                
        new_version = MediumPromptVersion(
            version=version,
            template_str=template_str,
            description=description
        )
        self._registry[name].append(new_version)
        print(f"Registered {name} v{version}")

    def get_prompt(self, name: str, version: Optional[str] = None) -> PromptTemplate:
        if name not in self._registry or not self._registry[name]:
            raise KeyError(f"Prompt '{name}' not found in registry.")
            
        versions = self._registry[name]
        
        if version is None:
            target_version = versions[-1]
        else:
            target_version = next((pv for pv in versions if pv.version == version), None)
            if target_version is None:
                raise ValueError(f"Version '{version}' not found for prompt '{name}'.")
                
        return PromptTemplate.from_template(target_version.template_str)

# Medium Example Usage
if __name__ == "__main__":
    med_registry = MediumPromptRegistry()
    med_registry.register("med_agent", "1.0.0", "Helpful assistant. {question}", "Initial")
    print(med_registry.get_prompt("med_agent").format(question="Hi?"))



Registered med_agent v1.0.0
Helpful assistant. Hi?


### Advanced Implementation
Production-grade with strict type hinting, docstrings, error handling, AI security (PII checks), and fallback mechanisms.

In [3]:
from typing import Dict, List, Optional
from pydantic import BaseModel, Field, field_validator
from langchain_core.prompts import PromptTemplate
import re
import logging

# Configure basic logging for the production-grade registry
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("PromptRegistry")

class AdvancedPromptVersion(BaseModel):
    """Advanced Implementation: Strict typing, PII security checks, and semver."""
    version: str = Field(..., description="Semantic version string (e.g., '1.0.0')")
    template_str: str = Field(..., description="The raw prompt template string")
    description: str = Field(..., description="Reason for this version or changes made")

    @field_validator('version')
    @classmethod
    def validate_version(cls, v: str) -> str:
        """Ensure valid semver format."""
        if not re.match(r"^\d+\.\d+\.\d+$", v):
            raise ValueError(f"Invalid semver: {v}")
        return v
        
    @field_validator('template_str')
    @classmethod
    def validate_pii_security(cls, v: str) -> str:
        """
        Security Check: Disallow raw PII variable requests in templates.
        This is a basic guardrail example against accidental PII exfiltration logic.
        """
        forbidden_terms = ["{ssn}", "{credit_card}", "{password}"]
        for term in forbidden_terms:
            if term in v.lower():
                raise ValueError(f"Security Policy Violation: Template cannot request {term}")
        return v

class AdvancedPromptRegistry:
    """
    Advanced Implementation: 
    Production-grade registry with logging, explicit error handling, 
    fallback mechanisms, and AI security best practices.
    """
    def __init__(self) -> None:
        self._registry: Dict[str, List[AdvancedPromptVersion]] = {}
        # Fallback template in case of severe missing prompt issues
        self._fallback_template = "You are a helpful AI assistant. Answer: {question}"

    def register(self, name: str, version: str, template_str: str, description: str) -> None:
        """Registers a prompt version safely."""
        try:
            if name not in self._registry:
                self._registry[name] = []
                
            for pv in self._registry[name]:
                if pv.version == version:
                    raise ValueError(f"Prompt '{name}' version '{version}' already exists.")
                    
            new_version = AdvancedPromptVersion(
                version=version,
                template_str=template_str,
                description=description
            )
            self._registry[name].append(new_version)
            logger.info(f"Successfully registered prompt '{name}' v{version}")
        except Exception as e:
            logger.error(f"Failed to register prompt {name} v{version}: {str(e)}")
            raise

    def get_prompt(self, name: str, version: Optional[str] = None) -> PromptTemplate:
        """
        Retrieves a PromptTemplate safely, utilizing a fallback mechanism 
        if the prompt is completely missing to avoid runtime crashes.
        """
        try:
            if name not in self._registry or not self._registry[name]:
                logger.warning(f"Prompt '{name}' not found. Using fallback mechanism.")
                return PromptTemplate.from_template(self._fallback_template)
                
            versions = self._registry[name]
            
            if version is None:
                # Retrieve latest by semver sorting
                # Assuming simple string sort works for basic X.Y.Z
                target_version = sorted(versions, key=lambda x: [int(p) for p in x.version.split('.')])[-1]
            else:
                target_version = next((pv for pv in versions if pv.version == version), None)
                if target_version is None:
                    logger.warning(f"Version '{version}' not found for '{name}'. Using latest.")
                    target_version = sorted(versions, key=lambda x: [int(p) for p in x.version.split('.')])[-1]
                    
            return PromptTemplate.from_template(target_version.template_str)
        except Exception as e:
            logger.critical(f"Critical error retrieving prompt '{name}': {str(e)}. Using fallback.")
            return PromptTemplate.from_template(self._fallback_template)

# Advanced Example Usage
if __name__ == "__main__":
    adv_registry = AdvancedPromptRegistry()
    
    # 1. Normal Registration
    adv_registry.register("secure_agent", "1.0.0", "Hello. Answer: {question}", "Initial version")
    
    # 2. PII Security Violation Registration (Will fail and be caught)
    try:
        adv_registry.register("secure_agent", "1.1.0", "What is their {ssn}?", "Bad version")
    except ValueError as e:
        print(f"Caught expected validation error: {e}")
        
    # 3. Safe Retrieval
    prompt = adv_registry.get_prompt("secure_agent", "1.0.0")
    print("Advanced Output:", prompt.format(question="Hi?"))
    
    # 4. Fallback Execution
    fallback_prompt = adv_registry.get_prompt("missing_agent")
    print("Fallback Output:", fallback_prompt.format(question="Hello?"))



2026-08-21 07:02:16,992 - PromptRegistry - INFO - Successfully registered prompt 'secure_agent' v1.0.0


2026-08-21 07:02:16,993 - PromptRegistry - ERROR - Failed to register prompt secure_agent v1.1.0: 1 validation error for AdvancedPromptVersion
template_str
  Value error, Security Policy Violation: Template cannot request {ssn} [type=value_error, input_value='What is their {ssn}?', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


2026-08-21 07:02:16,994 - PromptRegistry - WARNING - Prompt 'missing_agent' not found. Using fallback mechanism.


Caught expected validation error: 1 validation error for AdvancedPromptVersion
template_str
  Value error, Security Policy Violation: Template cannot request {ssn} [type=value_error, input_value='What is their {ssn}?', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
Advanced Output: Hello. Answer: Hi?
Fallback Output: You are a helpful AI assistant. Answer: Hello?


## Common Pitfalls in Production

1.  **Hardcoding Prompts in Application Logic:** Embedding multi-line f-strings deep inside controller functions makes it impossible for non-engineers (like PMs or Prompt Engineers) to iterate.
2.  **Lack of Regression Testing:** Changing a prompt to fix a bug for one specific query often breaks the model's performance on 10 other queries. You must run a golden dataset against new prompt versions before deployment.
3.  **Missing Telemetry:** If you log model responses but do not log the *exact prompt version* that generated them, you cannot correlate user feedback (thumbs up/down) to specific prompt changes.


## Practical Lab / Homework

**Task:** Extend the `PromptRegistry` to include a lightweight "Evaluation Tracker". 
1. Create a method to log an execution of a prompt version, including the input variables, output text, and a success boolean.
2. Provide a fully working script executing this tracking.
3. **Video Walkthrough:** Record a brief async video walkthrough of your design decisions, explaining your object-oriented approach and how you structured the evaluation tracker.

**Constraint:** No pseudo-code. Use strict type hinting.


In [4]:
import datetime
from typing import Dict, List, Optional
from pydantic import BaseModel, Field

class EvaluationLog(BaseModel):
    """Tracks the performance of a specific prompt execution."""
    prompt_name: str
    version: str
    inputs: Dict[str, str]
    output: str
    success: bool
    timestamp: datetime.datetime = Field(default_factory=datetime.datetime.now)

class EvalTrackingPromptRegistry(AdvancedPromptRegistry):
    """Extends the registry with evaluation tracking capabilities."""
    def __init__(self) -> None:
        super().__init__()
        self._evaluations: List[EvaluationLog] = []
        
    def log_evaluation(self, prompt_name: str, version: str, inputs: Dict[str, str], output: str, success: bool) -> None:
        """
        Logs the result of a prompt execution.
        """
        # Validate that the prompt exists
        self.get_prompt(prompt_name, version)
        
        log_entry = EvaluationLog(
            prompt_name=prompt_name,
            version=version,
            inputs=inputs,
            output=output,
            success=success
        )
        self._evaluations.append(log_entry)
        print(f"Logged evaluation for {prompt_name} v{version} (Success: {success})")
        
    def get_success_rate(self, prompt_name: str, version: str) -> float:
        """Calculates the success rate for a specific prompt version."""
        relevant_logs = [log for log in self._evaluations if log.prompt_name == prompt_name and log.version == version]
        if not relevant_logs:
            return 0.0
        successes = sum(1 for log in relevant_logs if log.success)
        return successes / len(relevant_logs)

# Lab Solution Execution
if __name__ == "__main__":
    eval_registry = EvalTrackingPromptRegistry()
    
    # Setup
    eval_registry.register(
        "sql_generator", 
        "1.0.0", 
        "Generate a SQL query for: {query}", 
        "Initial version"
    )
    
    # Simulate executions
    prompt = eval_registry.get_prompt("sql_generator", "1.0.0")
    
    # Execution 1 (Success)
    inputs1 = {"query": "Get all active users"}
    output1 = "SELECT * FROM users WHERE status = 'active';"
    eval_registry.log_evaluation("sql_generator", "1.0.0", inputs1, output1, success=True)
    
    # Execution 2 (Failure)
    inputs2 = {"query": "Drop the users table"}
    output2 = "DROP TABLE users;"
    eval_registry.log_evaluation("sql_generator", "1.0.0", inputs2, output2, success=False)
    
    # Check success rate
    rate = eval_registry.get_success_rate("sql_generator", "1.0.0")
    print(f"\nSuccess rate for sql_generator v1.0.0: {rate * 100}%")



2026-08-21 07:02:17,015 - PromptRegistry - INFO - Successfully registered prompt 'sql_generator' v1.0.0


Logged evaluation for sql_generator v1.0.0 (Success: True)
Logged evaluation for sql_generator v1.0.0 (Success: False)

Success rate for sql_generator v1.0.0: 50.0%


## Reference Links

- [LangChain Prompt Templates Documentation](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/)
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Pydantic Documentation](https://docs.pydantic.dev/latest/)
